# datasets

In [ ]:
from datasets import * 
dataset = load_dataset("madao33/new-title-chinese")
dataset

In [ ]:
dataset = load_dataset("madao33/new-title-chinese",split = "train[:50%]")
dataset 

In [ ]:
dataset = dataset["train"]
split_dataset = dataset.train_test_split(test_size = 0.1)
#  分类任务 stratify_by_column="label"
split_dataset

In [ ]:
split_dataset

In [ ]:
dataset["train"].select([0,1,2])

In [ ]:
dataset["train"]["title"][:10]

In [ ]:
filter_set = dataset.filter(lambda example: "中国" in example["title"])
filter_set["train"]["title"][:5]
filter_set

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese",local_file_only = True)
def process_fn(text):
    model_input = tokenizer(text["content"],max_length=512,truncation = True)
    label = tokenizer(text["title"],max_length=16,truncation = True)
    model_input["labels"]= label["input_ids"]
    return model_input

In [ ]:
processed_set = dataset.map(process_fn,batched=True)
processed_set

In [ ]:
def process_fn(text,tokenizer=tokenizer):
    model_input = tokenizer(text["content"],max_length=512,truncation = True)
    label = tokenizer(text["title"],max_length=16,truncation = True)
    model_input["labels"]= label["input_ids"]
    return model_input
processed_set = dataset.map(process_fn,num_proc=4)
processed_set

In [ ]:
processed_set = dataset.map(process_fn,batched=True,remove_columns=dataset["train"].column_names)
processed_set

In [ ]:
processed_set.save_to_disk("./processed_data")

In [ ]:
processed_set = load_from_disk("./processed_data")

## 加载本地数据集

In [ ]:
# 文件
dataset = load_dataset('csv',data_files=,split=)
dataset = Dataset.from_csv("")
# 文件夹
dataset = load_dataset('csv',data_dir= , split=)


# DataCollator

In [ ]:
from transformers import DataCollatorWithPadding

In [ ]:
dataset = load_dataset('csv',data_files="ChnSentiCorp_htl_all.csv",split= "train")
dataset = dataset.filter(lambda x : x["review"] is not None)
dataset

In [ ]:
def process_fn(text):
    model_input = tokenizer(text["review"],max_length=64,truncation = True)
    model_input["labels"]= text["label"]
    return model_input

In [ ]:
tokenized_dataset = dataset.map(process_fn,batched=True,remove_columns=dataset.column_names)
tokenized_dataset

In [ ]:
collator = DataCollatorWithPadding(tokenizer= tokenizer)

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
d = DataLoader(tokenized_dataset,batch_size=8,collate_fn=collator,shuffle = True)

In [ ]:
next(enumerate(d))

# 实践

In [1]:
from transformers import DataCollatorWithPadding,AutoTokenizer
from datasets import * 
from torch.utils.data import DataLoader

In [2]:
dataset = load_dataset('csv',data_files="ChnSentiCorp_htl_all.csv",split= "train")
dataset = dataset.filter(lambda x : x["review"] is not None)
dataset

Dataset({
    features: ['label', 'review'],
    num_rows: 7765
})

In [3]:
dataset = dataset.train_test_split(test_size=0.1)

In [5]:
model="lxyuan/distilbert-base-multilingual-cased-sentiments-student"
tokenizer = AutoTokenizer.from_pretrained(model,local_files_only = True)

In [6]:
def process_fn(text,tokenizer = tokenizer):
    model_input = tokenizer(text["review"],max_length=64,truncation = True)
    model_input["labels"]= text["label"]
    return model_input
tokenized_dataset = dataset.map(process_fn,batched=True,remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/6988 [00:00<?, ? examples/s]

Map:   0%|          | 0/777 [00:00<?, ? examples/s]

In [8]:
train_set, test_set = dataset["train"],dataset["test"]
train_loader = DataLoader(train_set,batch_size=32,shuffle=True,collate_fn=DataCollatorWithPadding(tokenizer=tokenizer))
test_loader = DataLoader(test_set,batch_size=32,shuffle=False,collate_fn=DataCollatorWithPadding(tokenizer=tokenizer))